In [4]:
import pandas as pd
import sys
import os
from sklearn import set_config
import numpy as np
from lifelines.statistics import logrank_test
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sklearn.model_selection import train_test_split
from typing import Tuple
from sklearn.model_selection import GroupKFold
set_config(display="text")  # displays text representation of estimators

from sksurv.util import Surv
sys.path.append(os.path.abspath("../../"))
from src.utils.ConvertTextToCsv import TextToCsv
from src.utils.Preprocessing import Preprocessor
from src.utils.cox_models import Cox_regression, p_values_Cox_regression, Cox_l2_regression
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [6]:
pp = Preprocessor()
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)
df_clinical_data = pd.read_csv("../../data/raw/brca_tcga_pub2015_clinical_data.tsv", sep='\t')
df_clinical_keep = pp.clean_columns_dataset(df_clinical_data)

list_df = pp.total_type_len_type_cancer(df_clinical_keep)
df_clinical_keep["Tumor-Cancer"] = list_df
df_mRNA_transformed = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem.txt")
df_rsem_z_scores = TextToCsv("../../data/raw/data_mrna_seq_v2_rsem_zscores_ref_all_samples.txt")
clean_df = pp.eliminate_nan_genes(df_rsem_z_scores, "Hugo_Symbol")



Luminal A: 330 - Total(%): 0.40
Luminal B: 81 - Total(%):0.10
HER2-enriched: 23 - Total(%):0.03
TNBC: 85 - Total(%)0.10 
UNK: 299 - Total(%) 0.37
Shape of the CSV: (20440, 819)
Shape of the CSV: (20440, 819)
Max NaN for gene: 333
Genes before: 817
Genes after: 817


In [ ]:
df_merged = pp.merge_datasets(df_clinical_keep, clean_df)
cols = ["Sample ID","Tumor-Cancer", "Overall Survival Status", "Overall Survival (Months)"] + list(expressions_genes_cols)
df_eliminated_zeros = pp.eliminate_zero_genes(df_merged, "Tumor-Cancer", threshold=0.8)
expressions_genes_cols = df_eliminated_zeros.iloc[1:20441].sample(2000, axis="columns")

df_tnbc = df_eliminated_zeros.loc[
    df_merged["Tumor-Cancer"].isin(["TNBC"]),
    cols
]

df_others = df_eliminated_zeros.loc[
    df_merged["Tumor-Cancer"].isin(["Luminal A", "Luminal B", "HER2-enriched"])
]



Genes before Treshold: 20452
count    20452.000000
mean         0.030755
std          0.315342
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         36.000000
dtype: float64
Threshold (>80% zeros): 654 samples
After the treshold: 20452


In [ ]:
expression_df = expressions_genes_cols.iloc[:, 2:20441]
expression_df

,0,1,2,3,4,5,6,7,8,9,...,20429,20430,20431,20432,20433,20434,20435,20436,20437,20438
0,1.3388,0.2126,0.7984,-0.6307,-1.0943,-0.2599,-0.3948,-1.8383,-1.3290,-0.3469,...,-1.1474,0.9710,-0.0862,1.0512,-1.1751,1.2986,1.2820,1.2721,1.0933,1.4154
1,0.9617,-1.3931,-1.0102,-0.9439,-1.0943,-0.3914,-1.0800,-0.3298,-0.7987,-0.7609,...,-0.0969,0.9781,0.1352,0.4584,-0.7000,-0.0725,0.1197,0.2366,0.6362,0.1171
2,-0.4243,0.8852,-1.1796,-1.9832,-1.0943,-0.8874,-0.3096,0.0113,-0.8011,-0.7670,...,0.2367,0.9591,0.6064,-0.3010,-0.1998,0.2344,-0.2345,-0.5735,0.3722,1.7307
3,0.8793,0.6859,0.9742,-1.9832,-1.0943,-0.8874,1.0341,1.0011,-1.3290,-2.1344,...,-0.0334,0.4110,-0.0316,-0.1638,0.2530,-0.0270,0.9739,-0.5520,-1.4958,-0.2273
4,-0.9445,-0.7096,-0.8101,-1.9832,-1.0943,-0.8874,-1.0366,-0.5986,-1.3290,-2.1344,...,-1.0349,0.7772,0.1452,-1.2870,-1.1531,1.2733,1.0212,0.3022,0.1937,1.2808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
812,0.4162,0.8454,1.7894,1.4423,-1.0943,-0.4773,-1.6291,0.2191,-1.3290,-2.1344,...,-1.2784,0.5572,-0.4102,-0.0483,-0.5140,-0.1075,0.8048,1.6141,-0.8232,0.8602
813,-1.2976,0.1904,1.0837,2.1139,-1.0943,-0.8874,0.5356,-0.5145,-1.3290,-2.1344,...,-0.3804,-0.6431,0.6654,-0.7937,-1.7090,-1.0234,0.4492,0.1828,-0.4502,-1.6680
814,-1.9144,0.9673,0.3726,-0.7149,-1.0943,0.2136,0.7802,-1.3552,-0.6818,-2.1344,...,-0.5411,0.0024,-0.9949,-0.6620,0.9661,-1.8732,1.4247,-1.4656,-0.4270,-0.2981
815,-0.2469,0.3802,1.0151,-0.6302,-1.0943,-0.8874,2.6166,0.0962,1.3141,-2.1344,...,-0.5945,-0.6402,0.7989,-0.2232,-1.4595,-0.7983,0.9234,0.4794,-0.6506,1.4342


In [ ]:

p_values = []

for genes in expression_df:
    stats, p = ttest_ind(
        df_tnbc[genes],
        df_others[genes]
        
    )

     Overall Survival (Months)    3821    2611    1354  16635   16535    8484  \
7                        31.77 -1.5192  1.0938  1.8663    NaN  0.1709  0.8313   
10                       19.19  0.6575  1.0031  0.3215    NaN -1.4411 -0.0262   
13                       18.00  1.1840 -1.8891 -1.5132    NaN -1.4411 -0.1976   
14                       78.35  1.3298 -0.6463 -0.8833    NaN -1.4411 -0.5913   
16                       73.78  0.2210 -1.0112  0.4156    NaN -1.4411 -0.1602   
..                         ...     ...     ...     ...    ...     ...     ...   
788                      85.09 -0.6751 -0.4530  0.1582    NaN -1.4411  0.8960   
797                      69.25  2.1902 -1.0231 -1.6854    NaN -1.4411  0.4045   
804                      32.72  0.8759  0.6493 -0.6084    NaN -1.4411  1.2867   
809                      14.45  0.4389 -1.1983 -1.3982    NaN -1.4411  0.1414   
814                      23.46  1.3115 -1.3390 -0.5092    NaN -1.4411 -0.8181   

       3292    7070    5071

KeyboardInterrupt: 

<font size="4">Cox Lasso</font>

In [2]:
cox_lasso = CoxnetSurvivalAnalysis(l1_ratio=1.0, alpha_min_ratio=0.01)

<font size="4">Cox Ridge</font>

In [4]:
#Cox_l2_regression(...)

<font size="4">Elastic Net</font>

In [ ]:
cox_elastic_net = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)
cox_elastic_net.fit(Xt, y)